# SASA

**Paper**: [Large Language Models Can Become Strong Self-Detoxifiers](https://openreview.net/pdf?id=jY5oml9fe9)

**Authors**: Ching-Yun Ko, Pin-Yu Chen, Payel Das, Youssef Mroueh, Soham Dan, Georgios Kollias, Subhajit Chaudhury, Tejaswini Pedapati, Luca Daniel

SASA (self-disciplined autoregressive sampling) is an output steering method, enabling the users to perform controlled decoding given any desirable value attributes. 

SASA leverages the contextual representations from an LLM to learn linear subspaces from labeled data, e.g. characterizing toxic v.s. non-toxic output in analytical forms. When auto-completing a response token-by-token, SASA dynamically tracks the margin of the current output to steer the generation away from the toxic subspace, by adjusting the autoregressive sampling strategy. 

In this demo, we show how SASA can be used to reduce the toxicity of sentences generated by an LLM.

## Method parameters

| parameter           | type            | description                                                                                                           |
| ------------------- | --------------- | --------------------------------------------------------------------------------------------------------------------- |
| `beta`              | `float`         | Scaling coefficient for value redistribution. Must be non-negative.                                                   |
| `wv_path`           | `Optional[str]` | Path to a saved probe. Must end with `.probe` (JSON) or `.pt` (legacy tensor) if provided.                            |
| `gen_wv_data_path`  | `Optional[str]` | Path to the value dataset, e.g. sentences with labeled toxicity.                                                      |
| `gen_wv_length`     | `Optional[int]` | Maximum number of samples used for preparing SASA steering if `wv_path` does not exist.                               |
| `gen_wv_batch_size` | `Optional[int]` | Batch size used for preparing SASA steering if `wv_path` does not exist. Must be non-negative if `wv_path` is `None`. |
| `max_candidates`    | `Optional[int]` | Cap on the surviving candidate set scored per decoding step (top-N by score). `None` scores every surviving token.    |

## Setup

If running this from a Google Colab notebook, please uncomment the following cell to install the toolkit. The following block is not necessary if running this notebook from a virtual environment where the package has already been installed.

In [1]:
# !git clone https://github.com/IBM/AISteer360.git
# %cd AISteer360

The following authentication steps may be necessary to access any gated models (after being granted access by Hugging Face). Uncomment the following if you need to log in to the Hugging Face Hub:

In [2]:
# !pip install python-dotenv
# from dotenv import load_dotenv
# import os

# load_dotenv()
# token = os.getenv("HUGGINGFACE_TOKEN")
# from huggingface_hub import login
# login(token=token)

## Example: Steering for reduced toxicity

In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from aisteer360.algorithms.core.steering_pipeline import SteeringPipeline
from aisteer360.algorithms.output_control.sasa.control import SASA
import warnings

warnings.filterwarnings('ignore', category=UserWarning)

MODEL_NAME = "openai-community/gpt2"

/dccstor/principled_ai/users/erikmiehling/AISteer360/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Downloading data

By default, the toxicity subspace is constructed using the Jigsaw dataset from Kaggle. To use `jigsaw_unintended_bias` you can either download it manually from Kaggle (https://www.kaggle.com/c/jigsaw-unintended-bias-in-toxicity-classification/data) or run the following cell using the Kaggle API (https://www.kaggle.com/docs/api). Either way, all files should be extracted to one folder, e.g. `'./tmp/Jigsaw_data/all_data.csv'`.

#### Automated download instructions (run this if you haven't manually downloaded the dataset)

To access your Kaggle token (for downloading data using the API tool), first sign in at [kaggle.com](https://www.kaggle.com). Then:
- Click your profile photo -> "Your Profile" -> "Settings"
- Scroll to API and click "Create New Token"
- Your browser immediately downloads `kaggle.json`

Place the json in the kaggle directory in root (typically `~/.config/kaggle/`) and execute the following script. 

**Note**: If you encounter an error 403 (permission error), please ensure that you have clicked "Join the competition" under the "Data" tab on the dataset homepage. 

In [4]:
import sys
!{sys.executable} -m ensurepip --upgrade
!{sys.executable} -m pip install --upgrade pip setuptools wheel
!{sys.executable} -m pip install kaggle

Looking in links: /tmp/tmp1_82040_


In [5]:
import os, glob, zipfile, shutil, pandas as pd
from pathlib import Path
from kaggle.api.kaggle_api_extended import KaggleApi

DATA_DIR = Path("tmp/Jigsaw_data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

api = KaggleApi(); api.authenticate()
api.competition_download_files(
    "jigsaw-unintended-bias-in-toxicity-classification",
    path=str(DATA_DIR),
    force=True,
    quiet=False
)

zip_path = glob.glob(str(DATA_DIR / "*.zip"))[0]
with zipfile.ZipFile(zip_path) as z:
    z.extractall(DATA_DIR)

train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")

label_paths = [
    p for p in (
        DATA_DIR / "test_public_expanded.csv",
        DATA_DIR / "test_private_expanded.csv",
        DATA_DIR / "test_labels.csv"
    ) if p.exists()
]
if label_paths:
    lbl = pd.concat([pd.read_csv(p) for p in label_paths])
    test = test.merge(lbl[["id", "toxicity"]], on="id", how="left")

out_csv = DATA_DIR / "all_data.csv"
pd.concat([train, test]).to_csv(out_csv, index=False)

# cleanup
os.remove(zip_path)
for p in DATA_DIR.iterdir():
    if p.resolve() != out_csv.resolve():
        (p.unlink() if p.is_file() else shutil.rmtree(p))


  0%|          | 0.00/723M [00:00<?, ?B/s]


  0%|          | 1.00M/723M [00:04<48:36, 260kB/s]


  0%|          | 2.00M/723M [00:13<1:26:07, 146kB/s]


  0%|          | 3.00M/723M [00:16<1:04:43, 195kB/s]


  1%|          | 4.00M/723M [00:21<1:00:11, 209kB/s]

...


 94%|█████████▍| 682M/723M [01:40<00:00, 77.9MB/s]


 96%|█████████▌| 693M/723M [01:40<00:00, 84.8MB/s]


 97%|█████████▋| 704M/723M [01:40<00:00, 90.5MB/s]


 99%|█████████▉| 715M/723M [01:40<00:00, 95.4MB/s]


100%|██████████| 723M/723M [01:40<00:00, 7.52MB/s]

### Creating the control

SASA requires contructing the value subspace prior to the steering. To prepare the subspace, users should specify the sample budget `gen_wv_length` for the step. By setting `gen_wv_length = 1000`, users ask to construct the subspace from only 1k samples. By default, the algorithm uses all samples available with `gen_wv_length = -1`. The parameter `gen_wv_batch_size` represents the batch size used during this step. Users may also adjust it according to their computational resources.
Below, `beta` is a positive scalar that represents the steering strength, with `0` replicating the original decoding behavior.

At each decoding step SASA scores the surviving candidate tokens with a model forward to measure their subspace margin, so the per-step cost grows with the size of that candidate set. The `max_candidates` argument caps the set to the top-N tokens by current score before scoring, which bounds the per-step memory and compute. We set `max_candidates = 50` here; leaving it as `None` scores every surviving token (the full vocabulary at this stage), which is expensive for a large-vocabulary model.

In [6]:
sasa = SASA(
    beta=10,
    gen_wv_length=100,
    gen_wv_batch_size=8,
    gen_wv_data_path="tmp/Jigsaw_data",
    max_candidates=50,
)

If value subspace is available, users can skip the above parameters (`beta`, `gen_wv_length`, `gen_wv_data_path`) and instead specifiy the path to the subspace via `wv_path`. 

In [7]:
# sasa = SASA(
#     beta=10,
#     wv_path="tmp/steer_wv.probe",
# )

### Creating the steering pipeline

We create a `SteeringPipeline` with the `SASA` control.

In [8]:
sasa_pipeline = SteeringPipeline(
    model_name_or_path=MODEL_NAME,
    controls=[sasa],
    device_map="cuda",
    hf_model_kwargs={"low_cpu_mem_usage": True},
)

Next we steer the pipeline (under the single SASA control). Note that since we have initialized the SASA control with the path to the toxicity data, as opposed to passing in a trained subspace, steering requires learning this subspace from the data. This is resource-heavy step (GPU required).

In [9]:
sasa_pipeline.steer()

/dccstor/principled_ai/users/erikmiehling/AISteer360/aisteer360/algorithms/output_control/sasa/control.py:148: DtypeWarning: Columns (32,36) have mixed types. Specify dtype option on import or set low_memory=False.
  dataset = pd.read_csv(csv_path)


The fitted subspace lives on the control as a `LinearProbe` (`sasa.probe`). We save it to disk so it can be reused later without refitting, by loading it via the `wv_path` argument. The `save` method writes a `.probe` JSON file, adding the extension if it is absent.

In [10]:
sasa.probe.save("tmp/steer_wv.probe")

After steering, inference can now be run on the pipeline for a given prompt. We define a prompt that attempts to induce toxic behavior in the model.

In [11]:
PROMPT = "I had someone throw me to the ground and say: "

Passing this prompt into the unsteered (base) model demonstrates a toxic continuation.

In [12]:
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

inputs = tokenizer(PROMPT, return_tensors="pt")
input_ids = inputs.input_ids
attention_mask = inputs.attention_mask

gen_params = {
    "max_new_tokens": 100,
    "temperature": 0.6,
    "top_p": 0.9,
    "do_sample": True,
    "repetition_penalty": 1.05,
}

baseline_outputs = model.generate(
    **inputs.to(model.device), 
    **gen_params
)

print("\nResponse (baseline):\n")
print(tokenizer.decode(baseline_outputs[0][len(inputs['input_ids'][0]):], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



Response (baseline):

 "This is what you're going through. You know, I'm just trying not to think about this."
It was my first time being treated like that in public as a young man who has never been told by anyone else how it feels to be raped or abused on camera. It's an experience of having people around me telling us they have no idea why we are doing this. And if there were any other reason for them to do so then those would be things our parents could talk


Compare this with the response of the base model when steered using SASA (via the steering pipeline).

In [13]:
steered_output_ids = sasa_pipeline.generate(
    input_ids=input_ids,
    attention_mask=attention_mask,
    runtime_kwargs={},
    **gen_params,
)

print("\nResponse (SASA):\n")
print(tokenizer.decode(steered_output_ids[0], skip_special_tokens=True))


Response (SASA):

 "Hey, if you don't like this guy I'm going home. If you want to come back it's good."
So that was my first year of college as a player in St. Louis and when we won our First-Team All-Big 12 pick (in part because they drafted us) at No 1 overall out there for next season by way or another team with an opportunity to be on your own is just so much fun. It makes no sense now. The Big Ten


Lastly, note that the beta parameter dictates the strength of the steering, and can thus be adjusted to control the degree of toxicity suppression in the generated response (importantly without having to relearn the subspace).

In [14]:
sasa = SASA(
    beta=0,
    wv_path="tmp/steer_wv.probe",  # the subspace saved in the preparation steps above
    max_candidates=50,
)

sasa_pipeline = SteeringPipeline(
    model_name_or_path=MODEL_NAME,
    controls=[sasa],
    device_map="cpu",
    hf_model_kwargs={"low_cpu_mem_usage": True},
)

sasa_pipeline.steer()

original_output_ids = sasa_pipeline.generate(
    input_ids=input_ids,
    attention_mask=attention_mask,
    runtime_kwargs={},
    **gen_params,
)

print(f"\nResponse (beta=0):\n")
print(tokenizer.decode(original_output_ids[0], skip_special_tokens=True))


Response (beta=0):

 "Hey, that's not good enough. What do you mean by bad?"
So I told him "well if it was a fight then there would be no need for them." And he said "Well what are you talking about? You're going to have some trouble fighting this guy with your sword in hand so we'll just go get help from one of our friends or something like that," because they were all really drunk at the time! So after being outdone myself (and having
